In [ ]:
import pandas as pd
from argopy import DataFetcher
import xarray as xr

argo_loader = DataFetcher()
ds = argo_loader.region([32, 80, 8, 26, 0, 2000, '2023-03-01', '2023-03-31']).to_xarray()
ds = ds.to_dataframe().reset_index()
columns_to_keep = [
    'PLATFORM_NUMBER', 
    'CYCLE_NUMBER', 
    'LATITUDE', 
    'LONGITUDE', 
    'TIME',
    'PRES', 
    'TEMP', 
    'PSAL'
]
ds.dropna(inplace=True)
ds = ds[columns_to_keep]
ds.head()

/home/rohith/Projects/Poseidon-RAG/Backend/.venv/bin/python: No module named pip


,PLATFORM_NUMBER,CYCLE_NUMBER,LATITUDE,LONGITUDE,TIME,PRES,TEMP,PSAL
0,6903046,303,23.494922,63.744607,2023-01-01 14:23:30,2.9,25.518999,36.617001
1,6903046,303,23.494922,63.744607,2023-01-01 14:23:30,3.8,25.518999,36.616001
2,6903046,303,23.494922,63.744607,2023-01-01 14:23:30,4.8,25.520000,36.616001
3,6903046,303,23.494922,63.744607,2023-01-01 14:23:30,5.8,25.520000,36.616001
4,6903046,303,23.494922,63.744607,2023-01-01 14:23:30,6.9,25.520000,36.616001


In [2]:
ds.info()

<class 'pandas.DataFrame'>
Index: 308228 entries, 0 to 321609
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   PLATFORM_NUMBER  308228 non-null  int64         
 1   CYCLE_NUMBER     308228 non-null  int64         
 2   LATITUDE         308228 non-null  float64       
 3   LONGITUDE        308228 non-null  float64       
 4   TIME             308228 non-null  datetime64[ns]
 5   PRES             308228 non-null  float32       
 6   TEMP             308228 non-null  float32       
 7   PSAL             308228 non-null  float32       
dtypes: datetime64[ns](1), float32(3), float64(2), int64(2)
memory usage: 17.6 MB


In [3]:
ds['TIME'].dt.month.nunique()

3

In [4]:
import os
import sqlite3

conn = sqlite3.connect(os.path.join(os.getcwd(),'data','argo_data.db'))
ds.to_sql('measurements',con=conn,index=False,if_exists='replace')

308228

In [5]:
query = '''
    SELECT 
        Platform_number,
        COUNT(*) as total_readings,
        ROUND(MIN(temp), 2) as min_temp,
        ROUND(MAX(temp), 2) as max_temp,
        ROUND(AVG(latitude), 4) as avg_lat,
        ROUND(AVG(longitude), 4) as avg_lon
    FROM measurements
    GROUP BY Platform_number
'''

query ="""
        select Count(distinct(Platform_number)) from measurements
        """

cursor = conn.cursor()
cursor.execute(query)

rows = cursor.fetchall()

print(rows)

[(20,)]
